# 07 — Robustness checks
Stress-tests the Family A vs. Family B comparison against three threats
identified in the audit:

1. **Outcome definition** — the primary target counts every
   non-employed respondent as "unemployed" (job-seekers, people not
   looking for work, homemakers, further study, occasional workers).
2. **Retrospective predictors** — the composite indices are rated in
   the same survey that reports employment, so they may be coloured by
   the outcome.
3. **Graduation cohort** — unemployment differs sharply by cohort, and
   cohorts may have been surveyed at different times after graduating.

**Protocol.** Every scenario uses repeated stratified 5-fold CV
(10 repeats) on *all available rows* (no 80/20 holdout), so scenarios
are comparable with each other. The reference scenario (primary target,
full feature set) is re-run under this same protocol; its numbers
differ slightly from notebooks 03–05, which use the 125-row training
split. All models are untuned, so no nested loop is needed.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import numpy as np
import pandas as pd
from IPython.display import display

from src.feature_engineering import (
    CANDIDATE_FEATURES, NUMERIC_FEATURES, NOMINAL_FEATURES, TARGET_COLUMN,
    PRE_OUTCOME_FEATURES, PRE_OUTCOME_NUMERIC, TARGET_VARIANTS, COHORT_COLUMN, POS_LABEL
)
from src.evaluation import (
    compare_models, pairwise_corrected_ttest, corrected_resampled_ttest,
    cv_split_sizes, N_SPLITS
)
from src.classification_models import build_baseline_models
from src.robustness import build_study_models, model_family, leave_one_group_out
from src.reporting import save_table

warnings.filterwarnings("ignore", category=UserWarning)

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "tracer_study_processed.csv")
print(df.shape)

(157, 15)


## Scenarios

In [2]:
SCENARIOS = {
    "reference (primary target, all features)":
        dict(target=TARGET_COLUMN, features=CANDIDATE_FEATURES, numeric=NUMERIC_FEATURES),
    "labour-force target":
        dict(target="employment_status_labour_force", features=CANDIDATE_FEATURES, numeric=NUMERIC_FEATURES),
    "occasional work = employed":
        dict(target="employment_status_occasional_employed", features=CANDIDATE_FEATURES, numeric=NUMERIC_FEATURES),
    "pre-outcome features only":
        dict(target=TARGET_COLUMN, features=PRE_OUTCOME_FEATURES, numeric=PRE_OUTCOME_NUMERIC),
}

summaries, raw = {}, {}
for name, sc in SCENARIOS.items():
    data = df.dropna(subset=[sc["target"]])
    X, y = data[sc["features"]], data[sc["target"]].astype(int)
    models = build_study_models(sc["numeric"], NOMINAL_FEATURES, y_train=y)
    summary, scores = compare_models(models, X, y)
    summaries[name] = summary.assign(n=len(y), n_pos=int((y == POS_LABEL).sum()))
    raw[name] = (scores, len(y))
    print(f"{name}: n={len(y)}, at-risk={int((y == POS_LABEL).sum())}")

reference (primary target, all features): n=157, at-risk=31


labour-force target: n=145, at-risk=19


occasional work = employed: n=157, at-risk=26


pre-outcome features only: n=157, at-risk=31


## PR-AUC and ROC-AUC by scenario

`base_rate` is PR-AUC's chance level for that scenario; ROC-AUC's is 0.5.

In [3]:
cols = ["n", "n_pos", "base_rate", "pr_auc", "pr_auc_corrected_ci_low",
        "pr_auc_corrected_ci_high", "roc_auc"]
robustness_summary = pd.concat(
    {name: s[cols] for name, s in summaries.items()}, names=["scenario", "model"]
).reset_index()
robustness_summary["family"] = robustness_summary["model"].map(model_family)
save_table(robustness_summary, "robustness_summary", index=False)

pr_table = robustness_summary.pivot(index="model", columns="scenario", values="pr_auc")[list(SCENARIOS)]
roc_table = robustness_summary.pivot(index="model", columns="scenario", values="roc_auc")[list(SCENARIOS)]
print("PR-AUC"); display(pr_table.round(3))
print("ROC-AUC"); display(roc_table.round(3))

PR-AUC


scenario,"reference (primary target, all features)",labour-force target,occasional work = employed,pre-outcome features only
model,,,,
A1_RandomForest_ClassWeighted,0.301,0.277,0.229,0.253
A2_XGBoost_ScalePosWeight,0.266,0.225,0.196,0.267
A3_SMOTE_RandomForest,0.295,0.288,0.255,0.258
A4_ADASYN_RandomForest,0.298,0.283,0.250,0.267
B1_IsolationForest,0.297,0.248,0.281,0.260
B2_OneClassSVM,0.267,0.240,0.238,0.298
B3_LocalOutlierFactor,0.301,0.263,0.270,0.295
BL1_Dummy_Prior,0.197,0.131,0.166,0.197
BL2_LogReg_Balanced,0.367,0.291,0.295,0.395


ROC-AUC


scenario,"reference (primary target, all features)",labour-force target,occasional work = employed,pre-outcome features only
model,,,,
A1_RandomForest_ClassWeighted,0.547,0.580,0.502,0.547
A2_XGBoost_ScalePosWeight,0.487,0.493,0.426,0.550
A3_SMOTE_RandomForest,0.539,0.604,0.528,0.535
A4_ADASYN_RandomForest,0.542,0.611,0.527,0.540
B1_IsolationForest,0.548,0.604,0.570,0.481
B2_OneClassSVM,0.460,0.497,0.431,0.504
B3_LocalOutlierFactor,0.512,0.558,0.514,0.500
BL1_Dummy_Prior,0.500,0.500,0.500,0.500
BL2_LogReg_Balanced,0.646,0.608,0.616,0.675


## Does the Family A vs. B conclusion hold in each scenario?

Per scenario: how many of the 12 cross-family pairs favour Family A,
how many are significant (corrected resampled t-test, Bonferroni across
all 21 study-model pairs), and whether the best Family A model beats the
logistic-regression baseline (BL2), and whether BL2 beats the
prior-only dummy (BL1), i.e. whether the outcome is predictable at all.

In [4]:
rows = []
for name, (scores, n) in raw.items():
    n_train, n_test = cv_split_sizes(n, N_SPLITS)
    study = {k: v for k, v in scores.items() if model_family(k) in ("A", "B")}
    pairs = pairwise_corrected_ttest(study, n_train, n_test)
    cross = pairs[pairs["model_a"].map(model_family) != pairs["model_b"].map(model_family)]
    a_minus_b = np.where(cross["model_a"].map(model_family) == "A", 1, -1) * cross["mean_diff"]

    best_a = max((k for k in study if model_family(k) == "A"), key=lambda k: study[k].mean())
    _, p_vs_lr = corrected_resampled_ttest(scores[best_a], scores["BL2_LogReg_Balanced"], n_train, n_test)
    _, p_lr_vs_dummy = corrected_resampled_ttest(scores["BL2_LogReg_Balanced"], scores["BL1_Dummy_Prior"], n_train, n_test)

    rows.append({
        "scenario": name,
        "cross_pairs_favouring_A": int((a_minus_b > 0).sum()),
        "cross_pairs_significant": int(cross["significant"].sum()),
        "min_cross_p": cross["p_value"].min(),
        "best_A": best_a,
        "best_A_minus_LogReg": study[best_a].mean() - scores["BL2_LogReg_Balanced"].mean(),
        "best_A_vs_LogReg_p": p_vs_lr,
        "LogReg_minus_Dummy": scores["BL2_LogReg_Balanced"].mean() - scores["BL1_Dummy_Prior"].mean(),
        "LogReg_vs_Dummy_p": p_lr_vs_dummy,
    })

conclusion_check = pd.DataFrame(rows)
display(conclusion_check.round(4))
save_table(conclusion_check, "robustness_conclusion_check", index=False);

,scenario,cross_pairs_favouring_A,cross_pairs_significant,min_cross_p,best_A,best_A_minus_LogReg,best_A_vs_LogReg_p,LogReg_minus_Dummy,LogReg_vs_Dummy_p
0,"reference (primary target, all features)",5,0,0.5996,A1_RandomForest_ClassWeighted,-0.0661,0.2838,0.1693,0.0094
1,labour-force target,9,0,0.6119,A3_SMOTE_RandomForest,-0.0030,0.9705,0.1603,0.0209
2,occasional work = employed,2,0,0.2477,A3_SMOTE_RandomForest,-0.0401,0.4950,0.1293,0.0231
3,pre-outcome features only,2,0,0.4766,A4_ADASYN_RandomForest,-0.1275,0.0488,0.1972,0.0045


## Graduation cohort

Cohort (`batch`) is not a modeling feature, but if unemployment depends
strongly on cohort — e.g. because cohorts were surveyed at different
times after graduating — models may partly be learning cohort
differences. Three checks:

1. Unemployment rate by cohort.
2. A cohort-only baseline: logistic regression on cohort alone, same CV.
3. Leave-one-cohort-out: train on two cohorts, score the third.

In [5]:
by_cohort = (
    df.groupby(COHORT_COLUMN)[TARGET_COLUMN]
    .agg(n="size", n_unemployed=lambda s: int((s == POS_LABEL).sum()))
)
by_cohort["unemployment_rate"] = by_cohort["n_unemployed"] / by_cohort["n"]
display(by_cohort.round(3))
save_table(by_cohort, "cohort_unemployment")

X_cohort = df[[COHORT_COLUMN]].astype(str)
y = df[TARGET_COLUMN].astype(int)
cohort_only = {"BL3_LogReg_CohortOnly": build_baseline_models([], [COHORT_COLUMN])["BL2_LogReg_Balanced"]}
cohort_summary, _ = compare_models(cohort_only, X_cohort, y)
display(cohort_summary[["pr_auc", "pr_auc_corrected_ci_low", "pr_auc_corrected_ci_high",
                        "roc_auc", "base_rate"]].round(3))
save_table(cohort_summary, "cohort_only_baseline");

,n,n_unemployed,unemployment_rate
batch,,,
2023,61,16,0.262
2024,54,2,0.037
2025,42,13,0.310


,pr_auc,pr_auc_corrected_ci_low,pr_auc_corrected_ci_high,roc_auc,base_rate
BL3_LogReg_CohortOnly,0.278,0.225,0.33,0.648,0.197


In [6]:
X, y = df[CANDIDATE_FEATURES], df[TARGET_COLUMN].astype(int)
loco = leave_one_group_out(
    build_study_models(NUMERIC_FEATURES, NOMINAL_FEATURES, y_train=y), X, y, df[COHORT_COLUMN]
)
save_table(loco, "leave_one_cohort_out", index=False)

print("PR-AUC on the held-out cohort (base rate in column header):")
headers = {g: f"{g} (base {r:.2f})" for g, r in
           loco.groupby("held_out_group")["base_rate"].first().items()}
display(loco.pivot(index="model", columns="held_out_group", values="pr_auc")
        .rename(columns=headers).round(3))
print("ROC-AUC on the held-out cohort:")
display(loco.pivot(index="model", columns="held_out_group", values="roc_auc").round(3))

PR-AUC on the held-out cohort (base rate in column header):


held_out_group,2023 (base 0.26),2024 (base 0.04),2025 (base 0.31)
model,,,
A1_RandomForest_ClassWeighted,0.302,0.051,0.441
A2_XGBoost_ScalePosWeight,0.274,0.049,0.351
A3_SMOTE_RandomForest,0.261,0.062,0.533
A4_ADASYN_RandomForest,0.280,0.059,0.466
B1_IsolationForest,0.282,0.037,0.544
B2_OneClassSVM,0.292,0.053,0.335
B3_LocalOutlierFactor,0.370,0.040,0.399
BL1_Dummy_Prior,0.262,0.037,0.310
BL2_LogReg_Balanced,0.311,0.085,0.624


ROC-AUC on the held-out cohort:


held_out_group,2023,2024,2025
model,,,
A1_RandomForest_ClassWeighted,0.529,0.490,0.656
A2_XGBoost_ScalePosWeight,0.452,0.452,0.565
A3_SMOTE_RandomForest,0.488,0.553,0.668
A4_ADASYN_RandomForest,0.500,0.510,0.662
B1_IsolationForest,0.497,0.250,0.629
B2_OneClassSVM,0.510,0.385,0.395
B3_LocalOutlierFactor,0.494,0.288,0.499
BL1_Dummy_Prior,0.500,0.500,0.500
BL2_LogReg_Balanced,0.599,0.692,0.753


## Split sensitivity

Notebooks 03–05 cross-validate on one 125-row training split
(`train_test_split(..., random_state=42)`) and score the 32-row holdout
once. This repeats that exact protocol for four other split seeds, to
show how much the headline ranking depends on which 125 rows were drawn.
`cv_pr_auc` is the 5x10 CV mean on the training split; `holdout_pr_auc`
is the model refit on that split and scored on its 32-row holdout
(6 at-risk graduates).

In [7]:
from sklearn.model_selection import train_test_split
from src.evaluation import pr_auc_scorer

SPLIT_SEEDS = [42, 0, 1, 2, 3]
X_all, y_all = df[CANDIDATE_FEATURES], df[TARGET_COLUMN].astype(int)

rows = []
for seed in SPLIT_SEEDS:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=0.2, random_state=seed, stratify=y_all
    )
    models = build_study_models(NUMERIC_FEATURES, NOMINAL_FEATURES, y_train=y_tr)
    summary, _ = compare_models(models, X_tr, y_tr)
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        rows.append({"split_seed": seed, "model": name, "family": model_family(name),
                     "cv_pr_auc": summary.loc[name, "pr_auc"],
                     "holdout_pr_auc": pr_auc_scorer(model, X_te, y_te)})

split_sensitivity = pd.DataFrame(rows)
save_table(split_sensitivity, "split_sensitivity", index=False)

print("CV PR-AUC on the 125-row training split, by split seed (42 = notebooks 03-05):")
display(split_sensitivity.pivot(index="model", columns="split_seed", values="cv_pr_auc")
        [SPLIT_SEEDS].round(3))
print("Holdout PR-AUC (32 rows, 6 at-risk), by split seed:")
display(split_sensitivity.pivot(index="model", columns="split_seed", values="holdout_pr_auc")
        [SPLIT_SEEDS].round(3))

CV PR-AUC on the 125-row training split, by split seed (42 = notebooks 03-05):


split_seed,42,0,1,2,3
model,,,,,
A1_RandomForest_ClassWeighted,0.426,0.359,0.302,0.279,0.260
A2_XGBoost_ScalePosWeight,0.386,0.328,0.274,0.273,0.211
A3_SMOTE_RandomForest,0.432,0.343,0.328,0.274,0.271
A4_ADASYN_RandomForest,0.440,0.347,0.318,0.266,0.268
B1_IsolationForest,0.287,0.348,0.292,0.309,0.309
B2_OneClassSVM,0.264,0.294,0.232,0.271,0.322
B3_LocalOutlierFactor,0.302,0.342,0.258,0.300,0.357
BL1_Dummy_Prior,0.200,0.200,0.200,0.200,0.200
BL2_LogReg_Balanced,0.385,0.410,0.345,0.339,0.334


Holdout PR-AUC (32 rows, 6 at-risk), by split seed:


split_seed,42,0,1,2,3
model,,,,,
A1_RandomForest_ClassWeighted,0.175,0.226,0.368,0.295,0.589
A2_XGBoost_ScalePosWeight,0.156,0.214,0.392,0.231,0.671
A3_SMOTE_RandomForest,0.150,0.193,0.273,0.270,0.592
A4_ADASYN_RandomForest,0.171,0.169,0.328,0.340,0.374
B1_IsolationForest,0.339,0.204,0.185,0.196,0.234
B2_OneClassSVM,0.365,0.166,0.326,0.259,0.195
B3_LocalOutlierFactor,0.289,0.158,0.284,0.260,0.204
BL1_Dummy_Prior,0.188,0.188,0.188,0.188,0.188
BL2_LogReg_Balanced,0.326,0.243,0.416,0.307,0.371
